# Chạy giao diện Streamlit hỏi đáp pháp luật trên Kaggle

Notebook này cài giao diện trong `demo/streamlit/`, đọc `OPENAI_API_KEY` và `NGROK_AUTH_TOKEN` từ Kaggle Secrets, rồi mở ứng dụng qua Ngrok. Không ghi secret ra output.

> **Đầu vào Kaggle:** corpus `legal-corpus.parquet`; model VietLegal-Harrier và multilingual-E5-large; output notebook fine-tune có checkpoint-1234; output notebook baseline có archive `*_complete_results.tar.gz` (LightGBM + Multiclass–Ridge). Cache BM25/dense là tùy chọn; thiếu cache thì notebook tự dựng index và lần khởi động đầu có thể lâu. Chọn GPU T4 trong phần Settings của notebook.

Bật Internet trong Kaggle để clone repo, tải tokenizer và cài package. Tạo hai Kaggle Secrets riêng: `OPENAI_API_KEY`, `NGROK_AUTH_TOKEN`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time
import urllib.request

REPO_URL = 'https://github.com/leeminsun1205/Drill-reliable-retrieval.git'
REPO_DIR = Path('/kaggle/working/Drill-reliable-retrieval')
APP_DIR = REPO_DIR / 'demo' / 'streamlit'
PORT = 8501
OPENAI_MODEL = 'gpt-4.1-mini'  # Có thể đổi theo model API mà tài khoản của bạn được phép dùng
NGROK_REGION = 'ap'
RETRIEVER_FACTORY = 'pipeline_adapter:create_backend'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (APP_DIR / 'app.py').is_file():
    raise FileNotFoundError(f'Không tìm thấy Streamlit app: {APP_DIR / "app.py"}')
print({'repo': str(REPO_DIR), 'app': str(APP_DIR), 'port': PORT, 'retriever_factory_configured': bool(RETRIEVER_FACTORY)})

In [ ]:
# Cài pipeline retrieval, VnCoreNLP, LightGBM, giao diện và pyngrok.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dense,vietnamese,ltr,demo]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok>=7,<8'], check=True)
vncore_dir = Path('/kaggle/working/VnCoreNLP')
vncore_jar = vncore_dir / 'VnCoreNLP-1.2.jar'
if not vncore_jar.is_file():
    if vncore_dir.exists():
        import shutil; shutil.rmtree(vncore_dir)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vncorenlp/VnCoreNLP.git', str(vncore_dir)], check=True)
if not vncore_jar.is_file():
    raise FileNotFoundError(f'Không tìm thấy VnCoreNLP JAR: {vncore_jar}')
print('Cài đặt hoàn tất.')

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
openai_key = secrets.get_secret('OPENAI_API_KEY')
ngrok_token = secrets.get_secret('NGROK_AUTH_TOKEN')
if not openai_key:
    raise RuntimeError('Thiếu Kaggle Secret OPENAI_API_KEY')
if not ngrok_token:
    raise RuntimeError('Thiếu Kaggle Secret NGROK_AUTH_TOKEN')
os.environ['OPENAI_API_KEY'] = openai_key
os.environ['OPENAI_MODEL'] = OPENAI_MODEL
if RETRIEVER_FACTORY:
    os.environ['DRILL_RETRIEVER_FACTORY'] = RETRIEVER_FACTORY
print('Đã nạp secrets vào môi trường tiến trình; không hiển thị giá trị.')

In [ ]:
# Kiểm tra các input và model artifact trước khi mở public tunnel.
sys.path.insert(0, str(APP_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))  # Editable install runs in a child pip process.
import pipeline_adapter as adapter
corpus_path = Path(os.environ.get('DRILL_CORPUS_PATH') or adapter._unique_file('legal-corpus.parquet'))
archive_path = Path(os.environ.get('DRILL_COMPLETE_RESULTS_ARCHIVE') or adapter._unique_file(adapter.ARTIFACT_GLOB))
checkpoint_path = Path(os.environ.get('DRILL_CHECKPOINT_PATH') or adapter._checkpoint_root())
harrier_path = Path(os.environ.get('DRILL_HARRIER_PATH') or adapter._model_root('vietlegal-harrier'))
e5_path = Path(os.environ.get('DRILL_E5_PATH') or adapter._model_root('multilingual-e5-large'))
vncore_jar = Path('/kaggle/working/VnCoreNLP/VnCoreNLP-1.2.jar')
if not corpus_path.is_file() or not vncore_jar.is_file():
    raise FileNotFoundError(f'Corpus or VnCoreNLP JAR missing: corpus={corpus_path}, jar={vncore_jar}')
extraction = adapter.WORK_ROOT / 'complete_results'
preflight_artifacts = adapter._load_artifacts(archive_path, extraction, checkpoint_path)
cache_hits = {name: hit for name, (_path, hit) in {
    'bm25': adapter._resolve_retrieval_cache('bm25s_vncorenlp'),
    'chunks': adapter._resolve_retrieval_cache('bge_m3_token_window448_chunks'),
    'harrier': adapter._resolve_retrieval_cache('vietlegal_harrier_0_6b_token_window448'),
    'e5': adapter._resolve_retrieval_cache('multilingual_e5_large_token_window448'),
}.items()}
print({'corpus': str(corpus_path), 'checkpoint': str(checkpoint_path), 'harrier': str(harrier_path), 'e5': str(e5_path), 'results_archive': str(archive_path), 'feature_count': len(preflight_artifacts.lightgbm_feature_names), 'cache_hits': cache_hits})

In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = ngrok_token
conf.get_default().region = NGROK_REGION
# Dọn tunnel cũ trong cùng phiên trước khi mở tunnel mới.
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

log_path = Path('/kaggle/working/streamlit.log')
log_file = log_path.open('w', encoding='utf-8')
streamlit_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', str(PORT),
     '--server.headless', 'true', '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    cwd=APP_DIR, env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT,
)

health_url = f'http://127.0.0.1:{PORT}/_stcore/health'
for attempt in range(60):
    if streamlit_process.poll() is not None:
        log_file.flush()
        raise RuntimeError(f'Streamlit dừng sớm. Xem log tại {log_path}')
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError(f'Streamlit chưa sẵn sàng. Xem log tại {log_path}')

public_url = ngrok.connect(PORT, 'http').public_url
print('Streamlit đang chạy:', public_url)
print('Giữ notebook và cell này chạy để duy trì ứng dụng.')
if not RETRIEVER_FACTORY:
    print('Lưu ý: giao diện mở được nhưng chat đang khóa vì chưa cấu hình retrieval adapter.')

## Dừng ứng dụng

Chạy cell dưới đây khi muốn dừng Streamlit và đóng các tunnel Ngrok của phiên notebook.

In [ ]:
try:
    ngrok.kill()
except Exception as error:
    print(f'Không đóng được tunnel: {error}')
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    try:
        streamlit_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        streamlit_process.kill()
if 'log_file' in globals():
    log_file.close()
print('Đã dừng ứng dụng.')